# Local Compactness and Global Repulsion: A Memory-Queue Framework for Imbalanced Skin Lesion Classification

**Mukut Chatterjee, Diptarko Bhattacharjee, Debatra Banerjee, Asya Lyanova, Dmitrii Kaplun, Ram Sarkar**  
*Proceedings of the 17th Indian Conference on Computer Vision, Graphics and Image Processing (ICVGIP 2026), ACM*

---

## Overview and Motivation

Early detection of skin malignancies through automated dermoscopic image classification has the potential to expand screening access and catch dangerous lesions before they metastasize. However, when we bring deep learning models into clinical practice, we immediately run into a fundamental challenge: real-world skin lesion data is severely long-tailed.

In benchmark cohorts like ISIC 2019, common benign conditions such as melanocytic nevi (`NV`) account for more than half of all collected images. Meanwhile, aggressive or rare malignancies—including dermatofibroma (`DF`), vascular lesions (`VASC`), squamous cell carcinoma (`SCC`), and actinic keratosis (`AK`)—appear only in small numbers.

Under standard cross-entropy training, deep neural networks naturally take the path of least resistance: they optimize average accuracy by predicting the dominant majority class, ignoring the rare conditions. In healthcare, this behavior is unacceptable—missing a rare melanoma or invasive carcinoma can be fatal.

Existing approaches to this dilemma usually run into one of two bottlenecks:
1. **Logit-Level Reweighting (Focal Loss, LDAM, Balanced Softmax):** These methods adjust loss penalties at the classifier head, but leave the underlying latent representations unconstrained. As a result, minority samples remain scattered and heavily overlapped with dominant classes in feature space.
2. **Batch-Level Metric and Contrastive Learning (SupCon, PaCo):** These frameworks rely on sampling positive and negative pairs within each mini-batch. When dealing with extreme imbalance, rare classes often have zero or only one instance in a batch of 128, leading to noisy gradients and unstable prototype estimation.

### Proposed Approach

We address these challenges by pairing a dual-scale metric learning objective with an Exponential Moving Average (EMA) memory queue ($M = 4096$):

- **Decoupled Feature Bank:** The persistent memory queue decouples representation statistics from the instantaneous batch size, giving minority classes a consistent pool of positive and negative samples throughout every training step.
- **Local Regularization ($\mathcal{L}_{\text{knn}}$):** Encourages local intra-class compactness by pulling samples closer to their top-$k$ nearest neighbors while enforcing an angular margin against the hardest negative in the queue.
- **Global Regularization ($\mathcal{L}_{\text{proto}}$):** Repels class prototypes globally across the unit hypersphere using a heavy-tailed Cauchy potential kernel, preventing gradient plateauing and cluster collapse.
- **Balanced Multi-Task Learning:** Combines effective-number weighted cross-entropy ($\mathcal{L}_{\text{WCE}}$) with smooth loss weight scheduling to guide early representation learning before tightening geometric boundaries.

## 1. Dataset Preparation & Stratified Partitioning

We evaluate our framework on the ISIC 2019 benchmark (25,331 dermoscopic images across eight diagnostic categories). Notice the severe distribution skew:
- `NV` (Melanocytic Nevus): 12,875 images (dominant majority, ~51%)
- `MEL` (Melanoma): 4,522 images
- `BCC` (Basal Cell Carcinoma): 3,323 images
- `BKL` (Benign Keratosis): 2,624 images
- `AK` (Actinic Keratosis): 867 images
- `SCC` (Squamous Cell Carcinoma): 628 images
- `VASC` (Vascular Lesions): 253 images (rare tail class, ~1.0%)
- `DF` (Dermatofibroma): 239 images (rare tail class, ~0.9%)

We use a stratified 80/10/10 split (training, validation, test) to ensure that minority proportions remain consistent across every split.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

DATASET_DIR = "/kaggle/input/datasets/salviohexia/isic-2019-skin-lesion-images-for-classification"

classes = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d)) and not d.startswith(".")
])

print("Classes:", classes)

image_paths = []
labels = []

for cls in classes:
    cls_dir = os.path.join(DATASET_DIR, cls)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(cls_dir, fname))
            labels.append(cls)

df = pd.DataFrame({"path": image_paths, "label": labels})
print("Total images:", len(df))
print("\nClass distribution:\n", df["label"].value_counts().sort_index())

train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42
)

print(f"\nSplit: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

## 2. Preprocessing & CLAHE Color Normalization

Dermoscopic images are acquired across multiple clinics with varying dermatoscope optics, lighting conditions, and immersion gels, causing uneven illumination gradients and lens vignetting.

Standard RGB histogram equalization tends to distort skin tones and washes out subtle erythematous or pigmented vascular structures. To avoid this, we transform images to the CIE $LAB$ color space and apply Contrast-Limited Adaptive Histogram Equalization (CLAHE) strictly to the Luminance ($L$) channel. The chromatic channels ($a, b$) remain untouched, preserving diagnostic color fidelity.

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image, ImageFile
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode

cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
ImageFile.LOAD_TRUNCATED_IMAGES = True

IMG_SIZE = 224

class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        img_np = np.array(img)
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        l = clahe.apply(l)
        lab = cv2.merge((l, a, b))
        return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1), interpolation=InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20, interpolation=InterpolationMode.BILINEAR),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10), ratio=(0.3, 3.3)),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

## 3. Dataset & DataLoader Pipelines

To ensure smooth data streaming across multiple worker processes without GIL-related worker stalls, file paths and mapped integer targets are stored in native Python lists.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class ISICDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        classes = sorted(self.df["label"].unique())
        self.class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        self.image_paths = self.df["path"].tolist()
        self.labels = [self.class_to_idx[l] for l in self.df["label"].tolist()]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
class_names = sorted(train_df["label"].unique())
label_map = {cls: idx for idx, cls in enumerate(class_names)}
idx_to_class = {idx: cls for cls, idx in label_map.items()}

for idx, cls in idx_to_class.items():
    print(f"[{idx}] {cls}")

df["label_idx"] = df["label"].map(label_map)

In [ ]:
import os

BATCH_SIZE = 128
NUM_WORKERS = min(2, os.cpu_count() or 1)

train_dataset = ISICDataset(train_df, train_transform)
val_dataset   = ISICDataset(val_df, val_test_transform)
test_dataset  = ISICDataset(test_df, val_test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0
)

print(f"DataLoaders ready: batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS}")

In [ ]:
images, labels = next(iter(train_loader))
print("Mini-batch shapes:", images.shape, labels.shape)

## 4. Class-Rebalanced Classification Loss ($\mathcal{L}_{\text{WCE}}$)

When classes are imbalanced by a 54:1 ratio, naive inverse frequency weighting ($w_c \propto 1/N_c$) severely over-penalizes majority classes and creates extreme gradient spikes on noisy minority samples.

Following Cui et al. (CVPR 2019), we calculate sample weights based on the effective number of samples, where parameter $\beta = 0.999$ accounts for visual sample redundancy in high-dimensional feature space:

$$w_c = \frac{1 - \beta}{1 - \beta^{n_c}}$$

$$\mathcal{L}_{\text{WCE}} = -\frac{1}{B}\sum_{i=1}^B w_{y_i}\log p(y_i \mid x_i)$$

This provides smooth, balanced weighting that supports rare classes without destabilizing optimization.

In [ ]:
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class_counts = train_df["label"].value_counts().sort_index().values
beta = 0.999
effective_num = 1.0 - np.power(beta, class_counts)
weights = (1.0 - beta) / effective_num
weights = weights / weights.sum() * len(weights)
weights = torch.tensor(weights, dtype=torch.float32).to(device)

print("Class counts :", class_counts)
print("Class weights:", weights)

## 5. Dynamic EMA Memory Queue ($\mathcal{Q}$)

In metric learning, contrastive pair generation is usually constrained by mini-batch size. For a rare category like `DF` (only ~0.9% of the dataset), a batch of 128 will contain zero instances roughly 38% of the time, causing contrastive updates for that class to stall.

To resolve this, we maintain an asynchronous feature queue $\mathcal{Q} = \{(q_j, \ell_j)\}_{j=1}^M$ with capacity $M = 4096$. When an incoming mini-batch arrives:
- If a queue entry matches the sample's ground-truth class, we apply exponential moving average smoothing ($\gamma = 0.99$):
  $$q_p \leftarrow \frac{\gamma q_p + (1-\gamma)\tilde{z}_i}{\|\gamma q_p + (1-\gamma)\tilde{z}_i\|_2}$$
- If the class differs or the slot is empty, the entry is directly overwritten.

This design guarantees that every mini-batch optimization step has access to historical representations from all eight diagnostic classes.

In [ ]:
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F

class EMAFeatureQueue:
    def __init__(self, feature_dim=512, queue_size=4096, momentum=0.99, num_classes=8, device="cpu"):
        self.queue_size = queue_size
        self.momentum = momentum
        self.feature_dim = feature_dim
        self.num_classes = num_classes
        self.device = device

        self.features = torch.zeros(queue_size, feature_dim, device=device)
        self.labels = torch.full((queue_size,), -1, dtype=torch.long, device=device)
        self.ptr = 0
        self.full = False
        self.ema_updates = 0
        self.direct_overwrites = 0

    @torch.no_grad()
    def enqueue(self, features, labels):
        features = F.normalize(features.detach(), dim=1)
        if features.size(1) != self.features.size(1):
            self.features = torch.zeros(self.queue_size, features.size(1), device=features.device)
            self.ptr = 0
            self.full = False

        B = features.size(0)
        for i in range(B):
            idx = self.ptr
            curr_lbl = labels[i].item()

            if self.labels[idx].item() == curr_lbl:
                self.features[idx] = self.momentum * self.features[idx] + (1.0 - self.momentum) * features[i]
                self.features[idx] = F.normalize(self.features[idx], dim=0)
                self.ema_updates += 1
            else:
                self.features[idx] = features[i]
                self.labels[idx] = labels[i]
                self.direct_overwrites += 1

            self.ptr += 1
            if self.ptr >= self.queue_size:
                self.ptr = 0
                self.full = True

    def get(self):
        if self.full:
            return self.features, self.labels
        return self.features[:self.ptr], self.labels[:self.ptr]

    def get_distribution_stats(self, class_names=None):
        valid_labels = self.labels if self.full else self.labels[:self.ptr]
        valid_labels = valid_labels[valid_labels >= 0].cpu().tolist()
        counts = Counter(valid_labels)
        n_cls = self.num_classes or 8
        class_counts = {c: counts.get(c, 0) for c in range(n_cls)}
        active_counts = [v for v in class_counts.values() if v > 0]

        total_updates = self.ema_updates + self.direct_overwrites
        ema_freq = (self.ema_updates / total_updates * 100.0) if total_updates > 0 else 0.0
        overwrite_freq = (self.direct_overwrites / total_updates * 100.0) if total_updates > 0 else 0.0

        min_entries = min(class_counts.values()) if len(active_counts) == n_cls else 0
        avg_entries = sum(class_counts.values()) / max(1, n_cls)
        coverage_pct = (len(active_counts) / max(1, n_cls)) * 100.0

        stats = {
            "total_entries": len(valid_labels),
            "min_entries": min_entries,
            "avg_entries": avg_entries,
            "minority_coverage_pct": coverage_pct,
            "ema_update_freq_pct": ema_freq,
            "direct_overwrite_freq_pct": overwrite_freq,
            "ema_updates_count": self.ema_updates,
            "direct_overwrites_count": self.direct_overwrites,
        }
        for c in range(n_cls):
            c_name = class_names[c] if (class_names is not None and c < len(class_names)) else f"class_{c}"
            stats[f"count_{c_name}"] = class_counts[c]
        return stats

## 6. Metric Learning Losses & Scheduling

Our total objective combines local intra-class grouping and global inter-class repulsion:

$$\mathcal{L} = \mathcal{L}_{\text{WCE}} + \lambda_{\text{knn}}(t)\,\mathcal{L}_{\text{knn}} + \lambda_{\text{proto}}(t)\,\mathcal{L}_{\text{proto}}$$

### Weighted $k$-Nearest Neighbours Loss ($\mathcal{L}_{\text{knn}}$)
For each mini-batch sample $\tilde{z}_i$, we query the queue for its $k=10$ nearest positive instances $\mathcal{P}_i$ and find the hardest negative neighbor in $\mathcal{N}_i$:

$$s_i^+ = \frac{1}{k}\sum_{j \in \text{top-}k(\mathcal{P}_i)} \langle\tilde{z}_i, q_j\rangle, \qquad s_i^- = \max_{j \in \mathcal{N}_i} \langle\tilde{z}_i, q_j\rangle$$

$$\mathcal{L}_{\text{knn}} = \frac{1}{B}\sum_{i=1}^B w_{y_i}\max\left(0,\, m - s_i^+ + s_i^-\right)$$

where angular margin $m = 0.20$ and $w_{y_i}$ prioritizes minority margin satisfaction.

### Cauchy Prototype Separation Loss ($\mathcal{L}_{\text{proto}}$)
Prototypes are computed as joint centroids combining live mini-batch embeddings and detached historical queue records:

$$v_c = \sum_{i \in \mathcal{B}_c} \tilde{z}_i + \sum_{j \in \mathcal{Q}_c} \text{sg}(q_j), \qquad \mu_c = \frac{v_c}{\|v_c\|_2 + \epsilon}$$

Pairwise prototype dispersion is evaluated using a heavy-tailed Cauchy potential kernel with bandwidth $\sigma = 0.5$:

$$\Phi(r, s) = \frac{1}{1 + \|\mu_r - \mu_s\|_2^2 / \sigma^2}, \qquad \mathcal{L}_{\text{proto}} = \frac{1}{C(C-1)}\sum_{r \neq s} \Phi(r, s)$$

Unlike standard Gaussian kernels whose gradients decay exponentially once centroids are moderately separated, the Cauchy kernel's polynomial power-law tail ($\mathcal{O}(d^{-2})$) maintains continuous repulsive pressure across the unit hypersphere throughout late training.

### Loss Weight Scheduling
To let the network establish coarse visual representations before enforcing rigid geometric boundaries, loss weights follow a smooth exponential warm-up:

$$\lambda(t) = \lambda_{\max} \cdot \frac{1 - e^{-t/\tau}}{1 - e^{-T/\tau}}$$

where $\tau_{\text{knn}} = 15, \tau_{\text{proto}} = 12, T = 60,$ and $\lambda_{\max} = 0.5$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TailAwareMarginKNNLoss(nn.Module):
    def __init__(self, class_weights, k=3, margin=0.2):
        super().__init__()
        self.k = k
        self.margin = margin
        self.register_buffer("class_weights", class_weights)

    def forward(self, features, labels, queue):
        features = F.normalize(features, dim=1)
        q_features, q_labels = queue.get()

        if len(q_labels) < 100:
            return features.sum() * 0.0, {}

        losses = []
        pos_sim_avg = 0.0
        neg_sim_avg = 0.0
        margin_ok = 0

        for i in range(features.size(0)):
            feat = features[i]
            label = labels[i]
            sim = feat @ q_features.T
            pos_mask = (q_labels == label)
            neg_mask = (q_labels != label)

            if pos_mask.sum() < self.k:
                continue

            pos_sims = sim[pos_mask]
            neg_sims = sim[neg_mask]
            pos_topk = torch.topk(pos_sims, self.k).values
            s_pos = pos_topk.mean()
            s_neg = neg_sims.max()

            loss_i = F.relu(self.margin - s_pos + s_neg) * self.class_weights[label]
            losses.append(loss_i)

            pos_sim_avg += float(s_pos.detach())
            neg_sim_avg += float(s_neg.detach())
            if s_pos > (s_neg + self.margin):
                margin_ok += 1

        if len(losses) == 0:
            return features.sum() * 0.0, {}

        total_loss = torch.stack(losses).mean()
        valid = len(losses)
        stats = {
            "pos_sim": pos_sim_avg / valid,
            "neg_sim": neg_sim_avg / valid,
            "margin_sat": 100.0 * margin_ok / valid
        }
        return total_loss, stats


class CauchyPrototypeLoss(nn.Module):
    def __init__(self, num_classes, sigma=0.5, min_samples=1):
        super().__init__()
        self.num_classes = num_classes
        self.sigma = sigma
        self.min_samples = min_samples

    def forward(self, features, labels, queue_features=None, queue_labels=None):
        if queue_features is not None and len(queue_labels) > 0:
            all_features = torch.cat([features, queue_features], dim=0)
            all_labels = torch.cat([labels, queue_labels], dim=0)
        else:
            all_features = features
            all_labels = labels

        prototypes = []
        valid_classes = 0
        for c in range(self.num_classes):
            mask = (all_labels == c)
            if mask.sum() < self.min_samples:
                continue
            proto = F.normalize(all_features[mask].mean(0), dim=0)
            prototypes.append(proto)
            valid_classes += 1

        if valid_classes < 2:
            return features.sum() * 0.0, {"proto_dist": 0.0, "proto_kernel": 0.0, "valid_classes": valid_classes}

        prototypes = torch.stack(prototypes, dim=0)
        dist_sq = ((prototypes[:, None] - prototypes[None]) ** 2).sum(-1)
        eye = torch.eye(valid_classes, device=dist_sq.device, dtype=torch.bool)
        pair_dist = dist_sq[~eye]

        kernel = 1.0 / (1.0 + pair_dist / (self.sigma ** 2))
        loss = kernel.mean()
        stats = {
            "proto_dist": pair_dist.mean().item(),
            "proto_kernel": kernel.mean().item(),
            "valid_classes": valid_classes
        }
        return loss, stats

In [ ]:
import math

def get_proto_lambda(epoch, total_epochs=60, max_lambda=0.5, tau=12):
    num = 1.0 - math.exp(-epoch / tau)
    den = 1.0 - math.exp(-total_epochs / tau)
    return max_lambda * num / den

def get_knn_lambda(epoch, total_epochs=60, max_lambda=0.5, tau=15):
    num = 1.0 - math.exp(-epoch / tau)
    den = 1.0 - math.exp(-total_epochs / tau)
    return max_lambda * num / den

## 7. Model Architecture (ResNet-18 with BNNeck)

When training deep networks with simultaneous classification and metric learning losses, an inherent tension emerges: Cross-Entropy encourages feature norms to grow towards infinity to saturate softmax probabilities, whereas metric losses require bounded embeddings on the unit hypersphere.

We resolve this by incorporating a BNNeck (Luo et al.):
1. A pretrained ResNet-18 extracts feature representations $z_i \in \mathbb{R}^{512}$.
2. A non-linear bottleneck ($512 \to 256 \to 512$) generates normalized representations $\tilde{z}_i = \frac{z_i}{\|z_i\|_2}$ used for the k-NN and prototype separation losses.
3. A `BatchNorm1d` layer (with bias frozen) normalizes embeddings prior to a linear classifier without bias, keeping classification logits decoupled from hyperspherical constraints.

In [ ]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResNet18KNNTimm(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.backbone = timm.create_model("resnet18", pretrained=True, num_classes=0, global_pool="avg")
        feat_dim = self.backbone.num_features

        for m in self.backbone.modules():
            if isinstance(m, nn.ReLU):
                m.inplace = False

        self.embedding = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=False),
            nn.Dropout(0.2),
            nn.Linear(256, feat_dim)
        )
        self.bnneck = nn.BatchNorm1d(feat_dim)
        self.bnneck.bias.requires_grad_(False)
        self.fc = nn.Linear(feat_dim, num_classes, bias=False)

    def forward(self, x):
        feat = self.backbone(x)
        emb = self.embedding(feat)
        emb_norm = F.normalize(emb, p=2, dim=1)
        cls_feat = self.bnneck(emb)
        logits = self.fc(cls_feat)
        return emb_norm, logits

## 8. Gradient Flow and Gradient Analysis

A natural question when introducing memory queues and prototype losses is whether the auxiliary objective actually transmits useful learning signals back to the feature encoder:
*Does differentiating the prototype loss with detached queue entries produce non-zero gradients for the model parameters? And does prototype repulsion conflict with cross-entropy classification?*

### Analytical Proof of Differentiability

Although historical queue embeddings $\mathcal{Q}_c$ are treated as detached records ($\text{sg}(q_j)$), active mini-batch embeddings $\tilde{z}_i(\theta) \in \mathcal{B}_c$ maintain complete autograd connections to the encoder.

Differentiating the prototype separation loss $\mathcal{L}_{\text{proto}}$ with respect to an active batch sample $\tilde{z}_i$ yields:

$$\frac{\partial \mathcal{L}_{\text{proto}}}{\partial \tilde{z}_i} \propto \frac{(\mathbf{I} - \mu_r \mu_r^\top)(\mu_r - \mu_s)}{\|v_r\|_2} \neq \mathbf{0}$$

The operator $(\mathbf{I} - \mu_r \mu_r^\top)$ acts as a projection matrix that ensures gradient updates remain tangential to the unit hypersphere, directly steering the encoder parameters $\theta$.

### Empirical Gradient Analysis Suite

During training, our engine monitors four diagnostic quantities at every optimization step:
1. **Quantity A ($G_p = \|\nabla_\theta \mathcal{L}_{\text{proto}}\|_2$):** The unweighted L2-norm of the prototype gradient with respect to trainable encoder parameters $\theta$. This confirms active backpropagation ($G_p > 0$) throughout all 60 epochs.
2. **Quantity B ($G_p^w = \lambda_{\text{proto}}(t) \cdot G_p$):** The effective weighted gradient magnitude scaled by the warm-up scheduler.
3. **Quantity C ($R_p = \frac{\|\lambda_{\text{proto}} \nabla \mathcal{L}_{\text{proto}}\|_2}{\|\nabla \mathcal{L}_{\text{total}}\|_2}$):** The relative contribution of the prototype loss to the total optimization vector, verifying that it maintains a stable, controlled share (~0.1% to 0.3%) without overwhelming classification.
4. **Quantity D ($\cos(\nabla \mathcal{L}_{\text{CE}}, \nabla \mathcal{L}_{\text{proto}})$):** The cosine alignment between the classification gradient and the prototype dispersion gradient:
   $$\cos(\theta) = \frac{\nabla_\theta \mathcal{L}_{\text{CE}} \cdot \nabla_\theta \mathcal{L}_{\text{proto}}}{\|\nabla_\theta \mathcal{L}_{\text{CE}}\|_2 \, \|\nabla_\theta \mathcal{L}_{\text{proto}}\|_2}$$
   - **Early Training (Epochs 1–10):** Alignment is positive ($\approx +0.15$ to $+0.20$), showing that prototype dispersion actively cooperates with cross-entropy in segregating clusters.
   - **Late Convergence (Epochs 50–60):** Alignment approaches zero ($\approx +0.01$ to $+0.02$), showing that prototype loss acts as an orthogonal geometric regularizer without gradient conflict.

In [ ]:
from sklearn.metrics import balanced_accuracy_score, f1_score
import torch

@torch.no_grad()
def validate(model, loader, ce_criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        _, logits = model(images)
        loss = ce_criterion(logits, labels)
        running_loss += loss.item()

        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / len(loader)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    acc = (torch.tensor(all_preds) == torch.tensor(all_labels)).float().mean().item()

    return {
        "loss": val_loss,
        "acc": acc * 100.0,
        "bal_acc": bal_acc * 100.0,
        "macro_f1": macro_f1 * 100.0
    }

In [ ]:
import math
import torch
from tqdm.auto import tqdm

NUM_EPOCHS = 60

def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    ce_criterion,
    knn_criterion,
    proto_criterion,
    queue,
    device,
    epoch,
    total_epochs
):
    model.train()
    running_loss = 0.0
    running_ce = 0.0
    running_knn = 0.0
    running_proto = 0.0

    running_proto_grad = 0.0
    running_proto_eff_grad = 0.0
    running_rel_contrib = 0.0
    running_cos_sim = 0.0

    total = 0
    correct = 0
    pos_sim_running = 0.0
    neg_sim_running = 0.0
    margin_running = 0.0
    knn_count = 0
    proto_step_count = 0

    lam_knn = get_knn_lambda(epoch, total_epochs=total_epochs, max_lambda=0.5)
    proto_lam = get_proto_lambda(epoch, total_epochs=total_epochs, max_lambda=0.5, tau=12)

    pbar = tqdm(
        loader,
        desc=f"Train Epoch [{epoch:02d}/{total_epochs:02d}]",
        dynamic_ncols=True,
        leave=True
    )

    for step, (images, labels) in enumerate(pbar, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            features, logits = model(images)
            ce_loss = ce_criterion(logits, labels)
            knn_loss, knn_stats = knn_criterion(features, labels, queue)

            q_features, q_labels = queue.get()
            proto_loss, proto_stats = proto_criterion(features, labels, q_features, q_labels)

            loss = ce_loss + lam_knn * knn_loss + proto_lam * proto_loss

        proto_grad_norm = 0.0
        proto_eff_grad_norm = 0.0
        grad_cos_sim = 0.0
        rel_grad_contrib = 0.0

        if proto_lam > 0 and proto_loss.requires_grad and proto_loss.item() > 0:
            trainable_params = [p for p in model.parameters() if p.requires_grad]

            proto_grads = torch.autograd.grad(
                proto_loss, trainable_params, retain_graph=True, allow_unused=True
            )
            ce_grads = torch.autograd.grad(
                ce_loss, trainable_params, retain_graph=True, allow_unused=True
            )

            dot_prod = 0.0
            p_sq_sum = 0.0
            ce_sq_sum = 0.0
            valid_p_grads = []

            for gp, gce in zip(proto_grads, ce_grads):
                if gp is not None:
                    valid_p_grads.append(gp.detach())
                    if gce is not None:
                        dot_prod += torch.sum(gp.detach() * gce.detach()).item()
                        ce_sq_sum += torch.sum(gce.detach() * gce.detach()).item()
                        p_sq_sum += torch.sum(gp.detach() * gp.detach()).item()

            if len(valid_p_grads) > 0:
                proto_grad_norm = torch.norm(torch.stack([torch.norm(g, 2) for g in valid_p_grads]), 2).item()
                proto_eff_grad_norm = proto_lam * proto_grad_norm

                if p_sq_sum > 0 and ce_sq_sum > 0:
                    denom = math.sqrt(p_sq_sum) * math.sqrt(ce_sq_sum)
                    grad_cos_sim = dot_prod / (denom + 1e-12)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        total_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        total_grad_val = float(total_grad_norm)

        if total_grad_val > 1e-8:
            rel_grad_contrib = proto_eff_grad_norm / total_grad_val

        if proto_grad_norm > 0:
            running_proto_grad += proto_grad_norm
            running_proto_eff_grad += proto_eff_grad_norm
            running_cos_sim += grad_cos_sim
            running_rel_contrib += rel_grad_contrib
            proto_step_count += 1

        scaler.step(optimizer)
        scaler.update()

        queue.enqueue(features, labels)

        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        running_loss += loss.item()
        running_ce += ce_loss.item()
        running_knn += knn_loss.item()
        running_proto += proto_loss.item()

        if len(knn_stats) > 0:
            pos_sim_running += knn_stats["pos_sim"]
            neg_sim_running += knn_stats["neg_sim"]
            margin_running += knn_stats["margin_sat"]
            knn_count += 1

        train_acc = 100.0 * correct / total
        postfix = {
            "loss": f"{running_loss/step:.3f}",
            "acc": f"{train_acc:.1f}%",
            "G_p": f"{proto_grad_norm:.3e}",
            "R_p": f"{rel_grad_contrib*100:.2f}%",
            "cos": f"{grad_cos_sim:+.3f}",
        }
        if knn_count > 0:
            postfix["margin"] = f"{margin_running/knn_count:.1f}%"
        pbar.set_postfix(postfix)

    valid_proto_steps = max(1, proto_step_count)
    epoch_stats = {
        "loss": running_loss / len(loader),
        "ce": running_ce / len(loader),
        "knn": running_knn / len(loader),
        "proto": running_proto / len(loader),
        "acc": 100.0 * correct / total,
        "knn_lambda": lam_knn,
        "proto_lambda": proto_lam,
        "proto_grad_norm": running_proto_grad / valid_proto_steps,
        "proto_eff_grad_norm": running_proto_eff_grad / valid_proto_steps,
        "rel_grad_contrib": running_rel_contrib / valid_proto_steps,
        "grad_cos_sim": running_cos_sim / valid_proto_steps,
    }

    if knn_count > 0:
        epoch_stats.update({
            "pos_sim": pos_sim_running / knn_count,
            "neg_sim": neg_sim_running / knn_count,
            "margin_sat": margin_running / knn_count
        })

    return epoch_stats

## 9. Model Training Execution

Training runs over 60 epochs using AdamW ($\text{lr} = 3 \times 10^{-4}$, weight decay $5 \times 10^{-4}$) scheduled via Cosine Annealing. Checkpointing is driven by validation Balanced Accuracy (macro-average recall) to ensure high sensitivity on minority malignancies.

In [ ]:
num_classes = 8
model = ResNet18KNNTimm(num_classes=num_classes).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: Total = {total_params:,}, Trainable = {trainable_params:,}")

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=weights,
    label_smoothing=0.15
)

knn_criterion = TailAwareMarginKNNLoss(
    class_weights=weights,
    k=10,
    margin=0.2
).to(device)

proto_criterion = CauchyPrototypeLoss(
    num_classes=num_classes,
    sigma=0.5,
    min_samples=1
)

queue = EMAFeatureQueue(
    feature_dim=512,
    queue_size=4096,
    momentum=0.99,
    num_classes=num_classes,
    device=device
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS
)

scaler = torch.amp.GradScaler("cuda")

In [ ]:
import numpy as np
import pandas as pd

best_bal_acc = 0.0
gradient_history = []
per_epoch_metrics = []
queue_distribution_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_stats = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        criterion,
        knn_criterion,
        proto_criterion,
        queue,
        device,
        epoch,
        NUM_EPOCHS
    )

    val_stats = validate(
        model,
        val_loader,
        criterion,
        device
    )

    gradient_history.append({
        "epoch": epoch,
        "loss": train_stats["loss"],
        "proto": train_stats["proto"],
        "proto_lambda": train_stats["proto_lambda"],
        "proto_grad_norm_G_p": train_stats["proto_grad_norm"],
        "proto_eff_grad_norm_G_p_w": train_stats["proto_eff_grad_norm"],
        "rel_grad_contrib_R_p": train_stats["rel_grad_contrib"],
        "grad_cos_sim": train_stats["grad_cos_sim"],
    })

    queue_stats = queue.get_distribution_stats(class_names=class_names)
    queue_stats["epoch"] = epoch
    queue_distribution_history.append(queue_stats)

    per_epoch_metrics.append({
        "epoch": epoch,
        "train_loss": train_stats["loss"],
        "train_ce": train_stats["ce"],
        "train_knn": train_stats["knn"],
        "train_proto": train_stats["proto"],
        "train_acc": train_stats["acc"],
        "knn_lambda": train_stats["knn_lambda"],
        "proto_lambda": train_stats["proto_lambda"],
        "val_loss": val_stats["loss"],
        "val_acc": val_stats["acc"],
        "val_bal_acc": val_stats["bal_acc"],
        "val_macro_f1": val_stats["macro_f1"],
    })

    scheduler.step()

    print(f"\nEpoch [{epoch:02d}/{NUM_EPOCHS:02d}]")
    print(f"Train Loss: {train_stats['loss']:.4f} (CE: {train_stats['ce']:.4f}, KNN: {train_stats['knn']:.4f}, Proto: {train_stats['proto']:.4f})")
    print(f"Val Loss  : {val_stats['loss']:.4f} | Val Acc: {val_stats['acc']:.2f}% | Bal Acc: {val_stats['bal_acc']:.2f}% | Macro F1: {val_stats['macro_f1']:.2f}%")
    print(f"G_p: {train_stats['proto_grad_norm']:.6f} | G_p^w: {train_stats['proto_eff_grad_norm']:.6f} | R_p: {train_stats['rel_grad_contrib']*100:.3f}% | cos: {train_stats['grad_cos_sim']:+.4f}")
    print(f"Queue: min={queue_stats['min_entries']} (DF), avg={queue_stats['avg_entries']:.1f}, coverage={queue_stats['minority_coverage_pct']:.1f}%")

    if val_stats["bal_acc"] > best_bal_acc:
        best_bal_acc = val_stats["bal_acc"]
        torch.save(model.state_dict(), "best_resnet18_knn_proto.pth")
        print(f"✓ Best Model Saved (Bal Acc = {best_bal_acc:.2f}%)")

pd.DataFrame(per_epoch_metrics).to_csv("training_metrics_per_epoch.csv", index=False)
pd.DataFrame(queue_distribution_history).to_csv("queue_distribution_per_epoch.csv", index=False)

per_10_epoch_summary = []
for i in range(0, NUM_EPOCHS, 10):
    chunk = gradient_history[i:i+10]
    q_chunk = queue_distribution_history[i:i+10]
    per_10_epoch_summary.append({
        "epoch_range": f"{i+1}-{i+10}",
        "mean_proto_loss": float(np.mean([e["proto"] for e in chunk])),
        "mean_proto_grad_G_p": float(np.mean([e["proto_grad_norm_G_p"] for e in chunk])),
        "mean_proto_eff_grad_G_p_w": float(np.mean([e["proto_eff_grad_norm_G_p_w"] for e in chunk])),
        "mean_rel_contrib_pct": float(np.mean([e["rel_grad_contrib_R_p"] for e in chunk]) * 100),
        "mean_cos_alignment": float(np.mean([e["grad_cos_sim"] for e in chunk])),
        "mean_queue_min_entries": float(np.mean([q["min_entries"] for q in q_chunk])),
        "mean_queue_avg_entries": float(np.mean([q["avg_entries"] for q in q_chunk])),
        "mean_queue_ema_freq_pct": float(np.mean([q["ema_update_freq_pct"] for q in q_chunk])),
    })

pd.DataFrame(per_10_epoch_summary).to_csv("metrics_per_10_epochs.csv", index=False)
print("Artifacts saved: training_metrics_per_epoch.csv, queue_distribution_per_epoch.csv, metrics_per_10_epochs.csv")

In [ ]:
import pandas as pd

for fname in ["training_metrics_per_epoch.csv", "metrics_per_10_epochs.csv", "queue_distribution_per_epoch.csv"]:
    try:
        df = pd.read_csv(fname)
        print(f"\n--- {fname} ---")
        display(df.head(5))
    except Exception as e:
        print(f"Error loading {fname}: {e}")

## 10. Experimental Results & Benchmark Comparison

All experiments were conducted on an NVIDIA Tesla T4 GPU and evaluated across five independent runs using stratified 80/10/10 splits.

### Benchmark Comparison with Loss Functions

| Loss Function | Venue / Source | ISIC 2018 Overall Acc (%) | ISIC 2018 Bal Acc (%) | ISIC 2018 Macro F1 (%) | ISIC 2019 Overall Acc (%) | ISIC 2019 Bal Acc (%) | ISIC 2019 Macro F1 (%) |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| Weighted CE | CVPR 2019 | $81.22 \pm 1.56$ | $75.44 \pm 1.72$ | $73.91 \pm 1.81$ | $77.10 \pm 1.61$ | $66.74 \pm 1.56$ | $65.61 \pm 1.63$ |
| Focal Loss | ICCV 2017 | $82.72 \pm 1.67$ | $76.84 \pm 1.73$ | $76.05 \pm 1.90$ | $76.27 \pm 1.51$ | $65.91 \pm 1.67$ | $66.19 \pm 1.64$ |
| LDAM | NeurIPS 2019 | $81.42 \pm 1.42$ | $75.42 \pm 1.65$ | $74.10 \pm 1.68$ | $77.77 \pm 1.72$ | $67.76 \pm 1.79$ | $64.67 \pm 1.78$ |
| Balanced Softmax | NeurIPS 2020 | $83.72 \pm 1.78$ | $77.54 \pm 1.79$ | $72.55 \pm 1.78$ | $77.30 \pm 1.49$ | $64.36 \pm 1.52$ | $64.29 \pm 1.61$ |
| SupCon | NeurIPS 2020 | $86.21 \pm 2.01$ | $78.09 \pm 1.96$ | **$78.09 \pm 1.98$** | $81.46 \pm 1.68$ | $70.73 \pm 1.91$ | $71.27 \pm 1.92$ |
| PaCo | ICCV 2021 | $86.31 \pm 1.79$ | $78.43 \pm 1.83$ | $77.01 \pm 1.80$ | $81.58 \pm 1.89$ | $71.19 \pm 2.04$ | $70.01 \pm 1.95$ |
| **Proposed Method** | **ICVGIP 2026** | **$87.41 \pm 2.03$** | **$79.32 \pm 1.87$** | $77.89 \pm 1.91$ | **$82.78 \pm 1.97$** | **$71.97 \pm 1.93$** | **$72.40 \pm 1.91$** |

### Per-Class Recall and Diagnostic Balance (ISIC 2019)

| Class | Description | SupCon | PaCo | Proposed Method | Difference vs Best Baseline |
|:---|:---|:---:|:---:|:---:|:---:|
| **NV** | Melanocytic Nevus (Majority ~51%) | $91.00 \pm 0.28$ | $90.35 \pm 0.31$ | **$92.20 \pm 0.25$** | +1.20% |
| **MEL** | Melanoma | $77.02 \pm 1.22$ | **$79.01 \pm 1.34$** | $77.61 \pm 1.15$ | -1.40% |
| **BCC** | Basal Cell Carcinoma | $74.12 \pm 1.45$ | $74.67 \pm 1.51$ | **$76.60 \pm 1.39$** | +1.93% |
| **BKL** | Benign Keratosis | $65.40 \pm 1.61$ | $65.97 \pm 1.47$ | **$67.80 \pm 1.52$** | +1.83% |
| **AK** | Actinic Keratosis | **$52.32 \pm 3.15$** | $51.48 \pm 3.01$ | $52.05 \pm 2.85$ | -0.27% |
| **SCC** | Squamous Cell Carcinoma | $78.25 \pm 3.52$ | $76.41 \pm 3.69$ | **$78.95 \pm 3.27$** | +0.70% |
| **VASC** | Vascular Lesion (Rare ~1.0%) | $57.63 \pm 5.18$ | $58.81 \pm 4.54$ | **$59.79 \pm 4.89$** | +0.98% |
| **DF** | Dermatofibroma (Rare ~0.9%) | $70.10 \pm 5.69$ | **$72.82 \pm 6.17$** | $70.76 \pm 5.42$ | -2.06% |
| **Balanced Accuracy** | Overall across classes | $70.73 \pm 1.91$ | $71.19 \pm 2.04$ | **$71.97 \pm 1.93$** | +0.78% |
| **Overall Accuracy** | Standard accuracy | $81.46 \pm 1.68$ | $81.58 \pm 1.89$ | **$82.78 \pm 1.97$** | +1.20% |

In [ ]:
import torch
import numpy as np
from sklearn.metrics import recall_score, balanced_accuracy_score, accuracy_score, f1_score

@torch.no_grad()
def evaluate_loader(model, data_loader, device):
    model.eval()
    all_preds = []
    all_targets = []

    for batch in data_loader:
        if isinstance(batch, (list, tuple)):
            images, labels = batch[0], batch[1]
        elif isinstance(batch, dict):
            images, labels = batch["image"], batch["label"]

        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        logits = outputs[1] if isinstance(outputs, tuple) else (outputs["logits"] if isinstance(outputs, dict) else outputs)

        if labels.ndim > 1:
            labels = torch.argmax(labels, dim=1)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    return {
        "overall_acc": accuracy_score(all_targets, all_preds) * 100.0,
        "per_class_recall": recall_score(all_targets, all_preds, average=None, zero_division=0) * 100.0,
        "macro_recall": np.mean(recall_score(all_targets, all_preds, average=None, zero_division=0) * 100.0),
        "balanced_acc": balanced_accuracy_score(all_targets, all_preds) * 100.0,
        "macro_f1": f1_score(all_targets, all_preds, average="macro", zero_division=0) * 100.0
    }

def print_results(title, metrics, class_labels):
    print(f"\n{'='*45}")
    print(f" {title.upper()} METRICS")
    print(f"{'='*45}")
    print(f"{'Class':<10} | {'Recall (%)':<12}")
    print(f"{'-'*25}")
    for name, rec in zip(class_labels, metrics["per_class_recall"]):
        print(f"{name:<10} | {rec:6.2f}%")
    print(f"{'-'*25}")
    print(f"Overall Acc : {metrics['overall_acc']:6.2f}%")
    print(f"Balanced Acc: {metrics['balanced_acc']:6.2f}%")
    print(f"Macro F1    : {metrics['macro_f1']:6.2f}%")
    print(f"{'='*45}\n")

eval_class_names = [idx_to_class[i] for i in range(len(idx_to_class))]

val_results = evaluate_loader(model, val_loader, device)
print_results("Validation Set", val_results, eval_class_names)

test_results = evaluate_loader(model, test_loader, device)
print_results("Test Set", test_results, eval_class_names)

## 11. Representation Analysis (t-SNE) & Embedding Quality Metrics

### Embedding Quality and Geometric Separation Metrics
We quantitatively assess geometric cluster quality on the unit hypersphere:

| Metric | ISIC 2018 (Proposed) | ISIC 2019 (Proposed) | Target Direction |
|:---|:---:|:---:|:---:|
| **Silhouette Index** | $0.1819 \pm 0.0130$ | $0.2363 \pm 0.0170$ | Higher ($\uparrow$) |
| **KNN Purity** | **$0.8187 \pm 0.0381$** | **$0.7684 \pm 0.0327$** | Higher ($\uparrow$) |
| **Intra-Class Cosine** | $0.7660 \pm 0.0315$ | $0.7878 \pm 0.0387$ | Higher ($\uparrow$) |
| **Inter-Class Cosine** | **$0.5305 \pm 0.0361$** | **$0.5422 \pm 0.0354$** | Lower ($\downarrow$) |
| **Margin** | **$0.2355 \pm 0.0231$** | **$0.2456 \pm 0.0229$** | Higher ($\uparrow$) |
| **Separation Ratio** | **$1.44 \pm 0.14$** | **$1.45 \pm 0.13$** | Higher ($\uparrow$) |

### 2D t-SNE Projection
Test representations are projected into 2D using PCA initialisation followed by t-SNE with perplexity 30, showing clean class separation without majority-class entrapment.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors

@torch.no_grad()
def extract_features(model, loader, device):
    model.eval()
    feats, labels, preds = [], [], []
    for images, targets in loader:
        images = images.to(device)
        features, logits = model(images)
        features = torch.nn.functional.normalize(features, p=2, dim=1)
        feats.append(features.cpu())
        labels.append(targets.cpu())
        preds.append(logits.argmax(dim=1).cpu())

    feats = torch.cat(feats).numpy()
    labels = torch.cat(labels).numpy()
    preds = torch.cat(preds).numpy()
    return feats, labels, preds

def knn_purity(features, labels, k=10):
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(features)
    _, idx = nbrs.kneighbors(features)
    idx = idx[:, 1:]
    return np.mean([np.mean(labels[idx[i]] == labels[i]) for i in range(len(features))])

def class_separation_stats(features, labels):
    classes = np.unique(labels)
    centroids = []
    intra_cos = []
    for c in classes:
        cls_feat = features[labels == c]
        centroid = cls_feat.mean(axis=0)
        centroid /= np.linalg.norm(centroid) + 1e-8
        centroids.append(centroid)
        intra_cos.append(np.mean(cls_feat @ centroid))

    centroids = np.stack(centroids)
    inter_cos = centroids @ centroids.T
    mask = ~np.eye(len(classes), dtype=bool)
    inter_vals = inter_cos[mask]

    return {
        "mean_intra_cos": np.mean(intra_cos),
        "mean_inter_cos": np.mean(inter_vals),
        "margin": np.mean(intra_cos) - np.mean(inter_vals)
    }

def evaluate_embedding(model, loader, device):
    features, labels, preds = extract_features(model, loader, device)
    sil = silhouette_score(features, labels)
    dbi = davies_bouldin_score(features, labels)
    chi = calinski_harabasz_score(features, labels)
    purity = knn_purity(features, labels, k=10)
    sep_stats = class_separation_stats(features, labels)

    print("Embedding Geometry Metrics:")
    print(f"  Silhouette Index   : {sil:.4f}")
    print(f"  Davies-Bouldin     : {dbi:.4f}")
    print(f"  Calinski-Harabasz  : {chi:.2f}")
    print(f"  KNN Purity (k=10)  : {purity*100:.2f}%")
    print(f"  Mean Intra Cosine  : {sep_stats['mean_intra_cos']:.4f}")
    print(f"  Mean Inter Cosine  : {sep_stats['mean_inter_cos']:.4f}")
    print(f"  Angular Margin     : {sep_stats['margin']:.4f}")

    return features, labels

def plot_tsne(features, labels, class_names=None):
    n_pca = min(50, features.shape[1], features.shape[0] - 1)
    feats_pca = PCA(n_components=n_pca, random_state=42).fit_transform(features)

    tsne = TSNE(
        n_components=2,
        perplexity=min(30, max(5, len(features) // 20)),
        learning_rate="auto",
        init="pca",
        max_iter=2000,
        random_state=42
    )
    emb_2d = tsne.fit_transform(feats_pca)

    plt.figure(figsize=(8, 8), dpi=300)
    classes = np.unique(labels)
    cmap = plt.cm.tab10

    for i, c in enumerate(classes):
        idx = (labels == c)
        c_label = class_names[c] if class_names is not None else str(c)
        plt.scatter(
            emb_2d[idx, 0], emb_2d[idx, 1],
            s=12, alpha=0.7, color=cmap(i % 10), label=c_label, edgecolors="none"
        )
        cx, cy = emb_2d[idx, 0].mean(), emb_2d[idx, 1].mean()
        plt.text(
            cx, cy, c_label, fontsize=10, weight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8, edgecolor=cmap(i % 10))
        )

    plt.xticks([])
    plt.yticks([])
    plt.title("t-SNE of Latent Representations", fontsize=12, pad=10)
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
    plt.tight_layout()
    plt.show()
    return emb_2d

In [ ]:
features, labels = evaluate_embedding(model, val_loader, device)
plot_tsne(features, labels, class_names=eval_class_names)